<a href="https://colab.research.google.com/github/achalapunase/MazewithMAPElites/blob/main/Lab_1_Interpretable_layers_interventions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --pre pytorch-concepts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.0/356.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 24.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch_concepts as pyc

# Annotated Tensors

Annotations give semantic meaning to the concept dimension of a tensor. They store the ordered labels, their cardinalities (1 for binary, >1 for categorical), their types (binary, categorical or continuous), and optional per-label metadata. This is what lets you refer to a concept by name instead of by column index.

In [ ]:
annotations = pyc.Annotations(
    labels=["smoking", "genotype", "tar"],
    cardinalities=[1, 3, 1],
    types=["binary", "categorical", "continuous"],
)

annotations

Annotations(labels=['smoking', 'genotype', 'tar'], states=[['0'], ['0', '1', '2'], ['0']], cardinalities=[1, 3, 1], types=['binary', 'categorical', 'continuous'], metadata=None, concept_space=False)

An AnnotatedTensor pairs a plain torch.Tensor with an Annotations describing its second axis (axis 1). You can then slice columns by concept name or by type. Any operation that leaves the concept axis unchanged keeps the annotation attached.

In [ ]:
tensor = pyc.AnnotatedTensor(
    data=torch.randn(10, 5),  # (batch_size, sum(cardinalities))
    annotation=annotations,
)
tensor

tensor([[ 9.0921e-01,  1.2238e+00, -4.2840e-01,  1.9545e-01, -1.1061e-03],
        [-4.2161e-01, -2.7336e+00, -2.3898e+00, -1.1519e+00, -1.4201e+00],
        [-5.7897e-01,  8.7542e-01,  9.1055e-02,  1.5487e-01, -2.6500e-01],
        [-3.9603e-01,  7.8781e-01,  8.3323e-01, -8.6084e-01, -3.1408e-01],
        [-1.2695e+00,  8.2383e-01, -3.1315e-01,  1.0988e-01, -9.7626e-01],
        [ 1.5117e+00, -1.0142e+00, -7.2976e-01,  1.4562e-01, -6.3140e-01],
        [-6.7759e-01,  1.8339e+00,  1.1282e+00, -7.4242e-01,  4.8401e-01],
        [-5.8209e-01, -2.8864e-01, -1.3309e+00,  1.5051e+00, -1.2803e-02],
        [ 6.9216e-01,  1.6833e+00,  9.1285e-01,  1.7138e-01, -1.1811e-01],
        [-4.7294e-01,  1.6229e+00,  7.6760e-01, -1.2053e-01,  3.2013e-01]])
# annotations(axis=1): ['smoking', 'genotype', 'tar']

In [ ]:
tensor["smoking"]  # slice by concept name

tensor([[ 0.9092],
        [-0.4216],
        [-0.5790],
        [-0.3960],
        [-1.2695],
        [ 1.5117],
        [-0.6776],
        [-0.5821],
        [ 0.6922],
        [-0.4729]])
# annotations(axis=1): ['smoking']

In [ ]:
tensor.split_by_type("binary")  # slice by concept type

tensor([[ 0.9092],
        [-0.4216],
        [-0.5790],
        [-0.3960],
        [-1.2695],
        [ 1.5117],
        [-0.6776],
        [-0.5821],
        [ 0.6922],
        [-0.4729]])
# annotations(axis=1): ['smoking']

# Layers

Layers are the heart of the Low-Level API. All layers extend BaseConceptLayer: they accept concept scores and/or embeddings as input (both optional) and always return concept scores. Passing Annotations instead of an integer for in_concepts or out_concepts makes the layer semantics-aware: it knows concept names, cardinalities, and types, and can annotate its output as an AnnotatedTensor.

Layer names follow the pattern <OperationType><InputType>To<OutputType>, so LinearEmbeddingToConcept maps embeddings to concepts and LinearConceptToConcept maps concepts to concepts:

In [ ]:
embedding_dims = 7
c_encoder = pyc.nn.LinearEmbeddingToConcept(
    in_embeddings=embedding_dims,
    out_concepts=annotations
)
c_encoder

LinearEmbeddingToConcept(
  (encoder): Linear(in_features=7, out_features=5, bias=True)
)

Passing Annotations instead of an integer for in_concepts or out_concepts makes the layer semantics-aware: it knows concept names, cardinalities, and types, and can annotate its output as an AnnotatedTensor.

In [ ]:
c_train = torch.randn(3, sum(annotations.cardinalities))

c_encoder.annotate(c_train)

tensor([[ 0.6672, -0.8596, -1.4694,  0.0210,  0.1911],
        [ 1.8280,  0.2229, -0.4008,  0.0943,  0.5701],
        [ 0.0091,  2.6199, -1.8939,  1.1503, -2.0329]])
# annotations(axis=1): ['smoking', 'genotype', 'tar']

# Concept Bottleneck Model

A CBM is an encoder feeding a predictor. During training, supervise both concept and task predictions:

In [ ]:
n_features = 10
embedding_dims = 7
concept_dims = sum(annotations.cardinalities)
task_dims = 1

embedding_encoder = torch.nn.Sequential(
    torch.nn.Linear(n_features, embedding_dims),
    torch.nn.LeakyReLU(),
)
c_encoder = pyc.nn.LinearEmbeddingToConcept(
    in_embeddings=embedding_dims,
    out_concepts=annotations
)
y_predictor = pyc.nn.LinearConceptToConcept(
    in_concepts=concept_dims,
    out_concepts=task_dims
)

model = torch.nn.ModuleDict({
    "embedding_encoder": embedding_encoder,
    "concept_encoder": c_encoder,
    "task_predictor": y_predictor
})

model

ModuleDict(
  (embedding_encoder): Sequential(
    (0): Linear(in_features=10, out_features=7, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
  )
  (concept_encoder): LinearEmbeddingToConcept(
    (encoder): Linear(in_features=7, out_features=5, bias=True)
  )
  (task_predictor): LinearConceptToConcept(
    (predictor): Linear(in_features=5, out_features=1, bias=True)
  )
)

In [ ]:
# data
import torch.nn.functional as F

batch_size = 100
x_train = torch.randn(batch_size, n_features)
c_smoking = x_train[:, 2] > 0
c_genotype_1 = x_train[:, 0] < -1
c_genotype_2 = (x_train[:, 0] > -1).bool() & (x_train[:, 0] < 1).bool()
c_genotype_3 = x_train[:, 0] > 1
c_genotype = torch.stack([c_genotype_1, c_genotype_2, c_genotype_3], axis=1)
c_tar = (x_train[:, 1] > 0).bool() & (x_train[:, 5] < 0).bool()
c_train = torch.cat([c_smoking.unsqueeze(1), c_genotype, c_tar.unsqueeze(1)], axis=1).float()
y_train = (c_smoking & c_tar).float().unsqueeze(1)

c_train = c_encoder.annotate(c_train)
c_train[:3]

tensor([[1., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [1., 0., 1., 0., 0.]])
# annotations(axis=1): ['smoking', 'genotype', 'tar']

In [ ]:
from sklearn.metrics import accuracy_score, r2_score

concept_reg = 0.5
n_epochs = 1000

optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
binary_loss_fn = torch.nn.BCEWithLogitsLoss()
categorical_loss_fn = torch.nn.CrossEntropyLoss()
continuous_loss_fn = torch.nn.MSELoss()

c_train_binary = c_train.split_by_type("binary")
c_train_categorical = c_train.split_by_type("categorical")
c_train_continuous = c_train.split_by_type("continuous")

model.train()
for epoch in range(n_epochs):
    optimizer.zero_grad()

    # generate concept predictions
    embeddings = model["embedding_encoder"](x_train)
    c_pred = model["concept_encoder"](embeddings=embeddings)

    # activate concepts depending on their type
    c_pred = model["concept_encoder"].annotate(c_pred)
    c_pred_binary = c_pred.split_by_type("binary")
    c_pred_categorical = c_pred.split_by_type("categorical")
    c_pred_continuous = c_pred.split_by_type("continuous")
    c_pred_activated = torch.cat([
        F.sigmoid(c_pred_binary),
        F.softmax(c_pred_categorical, dim=1),
        F.leaky_relu(c_pred_continuous)
    ], axis=1)
    y_pred = model["task_predictor"](concepts=c_pred_activated)

    # compute concept loss depending on concept type
    binary_loss = binary_loss_fn(c_pred_binary, c_train_binary)
    categorical_loss = categorical_loss_fn(c_pred_categorical, c_train_categorical)
    continuous_loss = continuous_loss_fn(c_pred_continuous, c_train_continuous)
    concept_loss = binary_loss + categorical_loss + continuous_loss

    # total loss
    task_loss = binary_loss_fn(y_pred, y_train)
    loss = concept_loss + concept_reg * task_loss

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        task_accuracy = accuracy_score(y_train, y_pred.detach() > 0.)
        concept_binary_accuracy = accuracy_score(c_train_binary, c_pred_binary > 0.)
        concept_categorical_accuracy = accuracy_score(c_train_categorical.argmax(axis=1), c_pred_categorical.argmax(axis=1))
        concept_continuous_r2 = r2_score(c_train_continuous, c_pred_continuous.detach().numpy())
        print(f"Epoch {epoch}: Loss {loss.item():.2f} | Task Acc: {task_accuracy:.2f} | Concept Binary Acc: {concept_binary_accuracy:.2f} | Concept Categorical Acc: {concept_categorical_accuracy:.2f} | Concept Continuous R2: {concept_continuous_r2:.2f}")


Epoch 0: Loss 2.43 | Task Acc: 0.08 | Concept Binary Acc: 0.26 | Concept Categorical Acc: 0.47 | Concept Continuous R2: -0.24
Epoch 100: Loss 0.59 | Task Acc: 0.92 | Concept Binary Acc: 0.99 | Concept Categorical Acc: 0.98 | Concept Continuous R2: 0.32
Epoch 200: Loss 0.34 | Task Acc: 0.92 | Concept Binary Acc: 1.00 | Concept Categorical Acc: 1.00 | Concept Continuous R2: 0.31
Epoch 300: Loss 0.28 | Task Acc: 0.92 | Concept Binary Acc: 1.00 | Concept Categorical Acc: 1.00 | Concept Continuous R2: 0.32
Epoch 400: Loss 0.25 | Task Acc: 0.92 | Concept Binary Acc: 1.00 | Concept Categorical Acc: 1.00 | Concept Continuous R2: 0.34
Epoch 500: Loss 0.23 | Task Acc: 0.92 | Concept Binary Acc: 1.00 | Concept Categorical Acc: 1.00 | Concept Continuous R2: 0.38
Epoch 600: Loss 0.20 | Task Acc: 0.92 | Concept Binary Acc: 1.00 | Concept Categorical Acc: 1.00 | Concept Continuous R2: 0.43
Epoch 700: Loss 0.17 | Task Acc: 0.92 | Concept Binary Acc: 1.00 | Concept Categorical Acc: 1.00 | Concept Conti

# Interventions

InterventionModule wraps any layer as a drop-in replacement. The replacement has the same call signature, but concepts are selectively modified at inference time.

A strategy decides how to intervene. Two kinds are supported:

Concept strategies (BaseConceptInterventionStrategy): override the layer’s output concept values — e.g. DoIntervention (set to a constant) or GroundTruthIntervention (set to ground-truth labels).

Mechanism strategies (BaseModuleInterventionStrategy): modify the layer’s weights and connections — e.g. PositiveWeightsIntervention (force positive weights, making the layer monotonic).

A policy decides which concepts are targeted — e.g. UniformPolicy (all), UncertaintyPolicy (least certain), GradientPolicy (most influential).

## Ground truth interventions

Ground truth interventions are used by domain experts to improve model performance

In [ ]:
x_corrupted = torch.randn_like(x_train) * 10

embeddings = model["embedding_encoder"](x_corrupted)
c_pred = model["concept_encoder"](embeddings=embeddings)

# activate concepts depending on their type
c_pred = model["concept_encoder"].annotate(c_pred)
c_pred_binary = c_pred.split_by_type("binary")
c_pred_categorical = c_pred.split_by_type("categorical")
c_pred_continuous = c_pred.split_by_type("continuous")
c_pred_activated = torch.cat([
    F.sigmoid(c_pred_binary),
    F.softmax(c_pred_categorical, dim=1),
    F.leaky_relu(c_pred_continuous)
], axis=1)
y_pred = model["task_predictor"](concepts=c_pred_activated)


task_accuracy = accuracy_score(y_train, y_pred.detach() > 0.)
concept_binary_accuracy = accuracy_score(c_train_binary, c_pred.split_by_type("binary") > 0.)
concept_categorical_accuracy = accuracy_score(c_train_categorical.argmax(axis=1), c_pred.split_by_type("categorical").detach().numpy().argmax(axis=1))
concept_continuous_r2 = r2_score(c_train_continuous, c_pred.split_by_type("continuous").detach().numpy())

print(f"Accuracy before intervention: ")
print(f"Task Acc: {task_accuracy:.2f} | Concept Binary Acc: {concept_binary_accuracy:.2f} | Concept Categorical Acc: {concept_categorical_accuracy:.2f} | Concept Continuous R2: {concept_continuous_r2:.2f}")

Accuracy before intervention: 
Task Acc: 0.66 | Concept Binary Acc: 0.54 | Concept Categorical Acc: 0.34 | Concept Continuous R2: -104.34


In [ ]:
concept_encoder_intervened = pyc.nn.InterventionModule(
    model["concept_encoder"],
    intervention_strategy=pyc.nn.GroundTruthIntervention(c_train),
    intervention_policy=pyc.nn.UniformPolicy(),
    out_concepts_to_intervene_on=["tar", "smoking"]
)
c_pred = concept_encoder_intervened(embeddings)
# activate concepts depending on their type
c_pred = model["concept_encoder"].annotate(c_pred)
c_pred_binary = c_pred.split_by_type("binary")
c_pred_categorical = c_pred.split_by_type("categorical")
c_pred_continuous = c_pred.split_by_type("continuous")
c_pred_activated = torch.cat([
    c_pred_binary,
    F.softmax(c_pred_categorical, dim=1),
    c_pred_continuous
], axis=1)
y_pred = model["task_predictor"](concepts=c_pred_activated)


task_accuracy = accuracy_score(y_train, y_pred.detach() > 0.)
concept_binary_accuracy = accuracy_score(c_train_binary, c_pred.split_by_type("binary") > 0.)
concept_categorical_accuracy = accuracy_score(c_train_categorical.argmax(axis=1), c_pred.split_by_type("categorical").detach().numpy().argmax(axis=1))
concept_continuous_r2 = r2_score(c_train_continuous, c_pred.split_by_type("continuous").detach().numpy())

print(f"Accuracy after intervention: ")
print(f"Task Acc: {task_accuracy:.2f} | Concept Binary Acc: {concept_binary_accuracy:.2f} | Concept Categorical Acc: {concept_categorical_accuracy:.2f} | Concept Continuous R2: {concept_continuous_r2:.2f}")


Accuracy after intervention: 
Task Acc: 0.98 | Concept Binary Acc: 1.00 | Concept Categorical Acc: 0.34 | Concept Continuous R2: 1.00


## Do interventions

Do interventions are used to estimate causal effects concepts on the downstream task

In [ ]:
concept_encoder_intervened = pyc.nn.InterventionModule(
    model["concept_encoder"],
    intervention_strategy=pyc.nn.DoIntervention(1.0),
    intervention_policy=pyc.nn.UniformPolicy(),
    out_concepts_to_intervene_on=["smoking"]
)
c_pred = concept_encoder_intervened(embeddings)
# activate concepts depending on their type
c_pred = model["concept_encoder"].annotate(c_pred)
c_pred_binary = c_pred.split_by_type("binary")
c_pred_categorical = c_pred.split_by_type("categorical")
c_pred_continuous = c_pred.split_by_type("continuous")
c_pred_activated_1 = torch.cat([
    c_pred_binary,
    F.softmax(c_pred_categorical, dim=1),
    F.leaky_relu(c_pred_continuous)
], axis=1)
y_pred_1 = model["task_predictor"](concepts=c_pred_activated_1)

concept_encoder_intervened = pyc.nn.InterventionModule(
    model["concept_encoder"],
    intervention_strategy=pyc.nn.DoIntervention(0.0),
    intervention_policy=pyc.nn.UniformPolicy(),
    out_concepts_to_intervene_on=["smoking"]
)
c_pred = concept_encoder_intervened(embeddings)
# activate concepts depending on their type
c_pred = model["concept_encoder"].annotate(c_pred)
c_pred_binary = c_pred.split_by_type("binary")
c_pred_categorical = c_pred.split_by_type("categorical")
c_pred_continuous = c_pred.split_by_type("continuous")
c_pred_activated_0 = torch.cat([
    c_pred_binary,
    F.softmax(c_pred_categorical, dim=1),
    F.leaky_relu(c_pred_continuous)
], axis=1)
y_pred_0 = model["task_predictor"](concepts=c_pred_activated_0)

c_pred_activated_1[:3], c_pred_activated_0[:3]

(tensor([[ 1.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00, -2.7213e-02],
         [ 1.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  9.6321e+00],
         [ 1.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00, -7.5057e-03]],
        grad_fn=<SliceBackward0>),
 tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00, -2.7213e-02],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  9.6321e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00, -7.5057e-03]],
        grad_fn=<SliceBackward0>))

We can estimate the causal effect on the downstream task by comparing the outcomes

In [ ]:
avg_treatment_effect = pyc.nn.functional.cace_score(
    y_pred_0,
    y_pred_1
)
avg_treatment_effect

tensor([0.8203], grad_fn=<SubBackward0>)

This shows that smoking concept has a positive causal effect on cancer predictions.